# ENNx BF16 CUDA-Oxide on a Colab T4

This notebook builds ENNx through Buck2 and exercises dense seeded perturbation of a JAX BF16 parameter buffer without NumPy or CuPy. Select a T4 GPU runtime before running it.

In [ ]:
import json
import platform
import subprocess
import sys

gpu = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,compute_cap,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
print(json.dumps({"python": platform.python_version(), "gpu": gpu}, indent=2))
assert sys.platform == "linux" and platform.machine() == "x86_64"
assert "T4" in gpu, f"Expected a T4 runtime, received: {gpu}"

In [ ]:
from pathlib import Path
import shutil
import tarfile
import urllib.request

JJ_VERSION = "0.41.0"
checkout = Path("/content/ennx")
if shutil.which("jj") is None:
    archive = Path("/tmp/jj.tar.gz")
    urllib.request.urlretrieve(
        f"https://github.com/jj-vcs/jj/releases/download/v{JJ_VERSION}/jj-v{JJ_VERSION}-x86_64-unknown-linux-musl.tar.gz",
        archive,
    )
    with tarfile.open(archive) as bundle:
        member = bundle.getmember("jj")
        member.mode = 0o755
        bundle.extract(member, "/usr/local/bin")
if not checkout.exists():
    subprocess.run(
        ["jj", "git", "clone", "--depth", "1", "--branch", "cuda",
         "https://github.com/Kvutza/ennx.git", str(checkout)],
        check=True,
    )
subprocess.run(["jj", "status"], cwd=checkout, check=True)

## Build the pinned toolchain

The first execution installs LLVM 21, nightly-2026-04-03, and CUDA-Oxide at the repository's pinned revision. The cached state makes later executions incremental.

In [ ]:
subprocess.run(
    [sys.executable, "ops/colab_cuda.py", "setup"],
    cwd=checkout,
    check=True,
)

## Build and verify the BF16 path

Buck2 compiles the Rust host extension and CUDA-Oxide device crate for `sm_75`, packages the CPython 3.12 wheel, and runs the JAX DLPack parity check on the T4.

In [ ]:
buck = [
    "./buck2w", "--isolation-dir", "cuda", "build", "//:cuda-parity",
    "--target-platforms", "//:linux-x86_64-platform",
    "--local-only", "--num-threads", "4", "--show-output",
]
subprocess.run(buck, cwd=checkout, check=True)

## Use the resident API from JAX

The base and candidate remain on the T4. JAX exports BF16 through DLPack, CUDA-Oxide computes each seeded direction and accumulation in FP32, then rounds the candidate once to BF16. `jax.dlpack.from_dlpack(tree)` returns a zero-copy JAX view owned by the Rust object.

In [ ]:
wheel_cmd = [
    "./buck2w", "--isolation-dir", "cuda", "build", "//:cuda-wheel",
    "--target-platforms", "//:linux-x86_64-platform",
    "--local-only", "--show-full-json-output",
]
wheel_output = subprocess.check_output(wheel_cmd, cwd=checkout, text=True)
wheel = next(iter(json.loads(wheel_output.strip().splitlines()[-1]).values()))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--force-reinstall", wheel],
    check=True,
)
print(wheel)

In [ ]:
import jax
import jax.numpy as jnp
from ennx.experimental import Bf16Tree

PARAMETERS = 1_000_000
base = jax.device_put(jnp.linspace(-1.0, 1.0, PARAMETERS, dtype=jnp.bfloat16))
leaves = [
    (101, 0, 400_000, 2.0e-3),
    (103, 400_000, 300_000, 1.0e-3),
    (107, 700_000, 300_000, 5.0e-4),
]
tree = Bf16Tree(base, leaves)
tree.materialize([(41, 0.25), (73, -0.125)])
candidate = jax.dlpack.from_dlpack(tree)
changed, finite, mean_step = jax.device_get((
    jnp.count_nonzero(candidate != base),
    jnp.all(jnp.isfinite(candidate)),
    jnp.mean(jnp.abs(candidate.astype(jnp.float32) - base.astype(jnp.float32))),
))
assert candidate.dtype == jnp.bfloat16
assert int(changed) == PARAMETERS and bool(finite)
print({"parameters": PARAMETERS, "changed": int(changed), "mean_step": float(mean_step)})

The million-parameter buffer is a fast correctness example. A pretrained MoE uses the same interface after its JAX parameter pytree is flattened into one contiguous BF16 device buffer. ENNx stores only the resident base, candidate, compact leaf metadata, and seeded direction terms; it does not materialize a full FP32 perturbation vector.